In [2]:
import os
import re
import time
import pickle
import hashlib
import sqlite3
import requests
import pandas as pd
from datetime import datetime
from collections import deque
from urllib.parse import urlparse, urljoin
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer

print("Import thư viện thành công!")

Import thư viện thành công!


### Cấu hình Crawler

In [3]:
TOPIC = "Scientific Papers"
SEED_URL = "https://arxiv.org/list/cs/recent"
ALLOWED_DOMAINS = ["arxiv.org", "www.arxiv.org"]
MAX_DEPTH = 3
MAX_PAGES = 3000
REQUEST_TIMEOUT = 10
CRAWL_DELAY = 1.0
DB_PATH = "data/arxiv_crawler.db"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

print("=== CRAWLER CONFIGURATION ===")
print(f"Topic          : {TOPIC}")
print(f"Seed URL       : {SEED_URL}")
print(f"Allowed Domains: {ALLOWED_DOMAINS}")
print(f"Max Depth      : {MAX_DEPTH} | Max Pages: {MAX_PAGES}")
print(f"Crawl Delay    : {CRAWL_DELAY}s")

=== CRAWLER CONFIGURATION ===
Topic          : Scientific Papers
Seed URL       : https://arxiv.org/list/cs/recent
Allowed Domains: ['arxiv.org', 'www.arxiv.org']
Max Depth      : 3 | Max Pages: 3000
Crawl Delay    : 1.0s


### 3 URL Frontier & Duplicate Detector

In [4]:
class URLFrontier:
    def __init__(self, max_depth=2):
        self.queue = deque()
        self.visited = set()
        self.max_depth = max_depth
        self.discovered_count = 0
        self.skipped_count = 0

    def add_url(self, url, depth):
        self.discovered_count += 1
        if depth > self.max_depth or url in self.visited:
            self.skipped_count += 1
            return False
        self.visited.add(url)
        self.queue.append((url, depth))
        return True

    def get_next(self):
        return self.queue.popleft() if self.queue else None

    def is_empty(self):
        return len(self.queue) == 0

class DuplicateDetector:
    def __init__(self):
        self.fingerprints = set()

    def is_duplicate(self, text_content):
        if not text_content:
            return True
        fp = hashlib.sha256(text_content.strip().encode("utf-8")).hexdigest()
        if fp in self.fingerprints:
            return True
        self.fingerprints.add(fp)
        return False

frontier = URLFrontier(max_depth=MAX_DEPTH)
detector = DuplicateDetector()
print("Đã khởi tạo URL Frontier và Duplicate Detector.")

Đã khởi tạo URL Frontier và Duplicate Detector.


### 4 Database (Tạo bảng lưu bài báo & bảng lưu Model)

In [5]:
class Database:
    def __init__(self, db_path):
        os.makedirs(os.path.dirname(db_path), exist_ok=True)
        self.conn = sqlite3.connect(db_path)
        self.create_tables()

    def create_tables(self):
        cursor = self.conn.cursor()
        # Bảng 1: Pages lưu dữ liệu bài báo đã xử lý (Cấu trúc chuẩn theo form)
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS pages (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                url TEXT UNIQUE,
                domain TEXT,
                title TEXT,
                abstract TEXT,
                authors TEXT,
                content TEXT,
                is_article INTEGER DEFAULT 0,
                depth INTEGER,
                status_code INTEGER,
                crawled_at TEXT
            )
        ''')
        # Bảng 2: Links lưu đồ thị liên kết
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS links (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                source_url TEXT,
                target_url TEXT
            )
        ''')
        # Bảng 3: Lưu Model ML/NLP dạng nhị phân BLOB
        cursor.execute('''
            CREATE TABLE IF NOT EXISTS trained_models (
                model_name TEXT PRIMARY KEY,
                model_type TEXT,
                vocabulary_size INTEGER,
                created_at TEXT,
                model_data BLOB
            )
        ''')
        
        # Thêm Index giống file mẫu để tối ưu truy vấn
        cursor.execute('CREATE INDEX IF NOT EXISTS idx_pages_url ON pages(url)')
        cursor.execute('CREATE INDEX IF NOT EXISTS idx_links_source ON links(source_url)')
        
        self.conn.commit()

    def insert_page(self, url, domain, title, abstract, authors, content, is_article, depth, status_code, crawled_at):
        cursor = self.conn.cursor()
        cursor.execute('''
            INSERT OR IGNORE INTO pages (url, domain, title, abstract, authors, content, is_article, depth, status_code, crawled_at)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (url, domain, title, abstract, authors, content, is_article, depth, status_code, crawled_at))
        self.conn.commit()

    def insert_link(self, source_url, target_url):
        cursor = self.conn.cursor()
        cursor.execute('''
            INSERT INTO links (source_url, target_url)
            VALUES (?, ?)
        ''', (source_url, target_url))
        self.conn.commit()

    def insert_model(self, name, m_type, vocab_size, model_bytes):
        cursor = self.conn.cursor()
        cursor.execute('''
            INSERT OR REPLACE INTO trained_models (model_name, model_type, vocabulary_size, created_at, model_data)
            VALUES (?, ?, ?, ?, ?)
        ''', (name, m_type, vocab_size, datetime.now().isoformat(), sqlite3.Binary(model_bytes)))
        self.conn.commit()

    def close(self):
        self.conn.close()

db = Database(DB_PATH)
print("Đã kết nối SQLite Database (Cấu trúc mới).")

Đã kết nối SQLite Database (Cấu trúc mới).


### 5 Parser và Tiền xử lý dữ liệu

In [6]:
IGNORED_EXTENSIONS = ('.pdf', '.ps', '.gz', '.tar', '.zip', '.jpg', '.png', '.css', '.js')

def clean_text(text):
    if not text:
        return ""
    text = re.sub(r'\$.*?\$', '', text)
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_page_data(html_content, url):
    soup = BeautifulSoup(html_content, "html.parser")
    title = soup.title.get_text(strip=True) if soup.title else "No Title"
    authors = "Unknown"
    raw_abstract = ""

    if "/abs/" in url:
        title_tag = soup.find("h1", class_="title")
        if title_tag:
            title = title_tag.get_text(strip=True).replace("Title:", "").strip()

        authors_div = soup.find("div", class_="authors")
        if authors_div:
            authors = ", ".join([a.get_text(strip=True) for a in authors_div.find_all("a")])

        abstract_tag = soup.find("blockquote", class_="abstract")
        if abstract_tag:
            raw_abstract = abstract_tag.get_text(strip=True).replace("Abstract:", "").strip()

    clean_abstract = clean_text(raw_abstract if raw_abstract else title)
    return title, authors, raw_abstract, clean_abstract, soup

def extract_and_filter_links(soup, base_url, allowed_domains):
    valid_links = []
    for tag in soup.find_all("a", href=True):
        href = tag["href"].strip()
        if href.startswith(('mailto:', 'javascript:', 'tel:', '#')):
            continue

        full_url = urljoin(base_url, href)
        parsed = urlparse(full_url)

        if parsed.scheme not in ('http', 'https'):
            continue
        if parsed.netloc.lower() not in allowed_domains:
            continue
        if any(parsed.path.lower().endswith(ext) for ext in IGNORED_EXTENSIONS):
            continue
            
        # ==========================================
        # BỘ LỌC MỚI: Chỉ cho phép trang bài báo (/abs/) hoặc trang danh sách (/list/)
        # ==========================================
        if not (parsed.path.startswith('/abs/') or parsed.path.startswith('/list/')):
            continue
        
        # Bỏ qua các định dạng trang không cần thiết của bài báo (như /format/, /html/)
        if parsed.path.startswith('/format/') or parsed.path.startswith('/html/'):
            continue

        clean_url = f"{parsed.scheme}://{parsed.netloc}{parsed.path}".rstrip('/')
        if not clean_url:
            clean_url = f"{parsed.scheme}://{parsed.netloc}/"
        valid_links.append(clean_url)
        
    return valid_links

print("Đã thiết lập Parser và Preprocessing (Cập nhật chặn trang rác).")

Đã thiết lập Parser và Preprocessing (Cập nhật chặn trang rác).


### 6 Chạy Crawler & In thống kê theo form mẫu

In [7]:
crawled_count = 0
failed_requests = 0
status_code_stats = {}
depth_stats = {d: 0 for d in range(MAX_DEPTH + 1)}

session = requests.Session()
session.headers.update(HEADERS)

# Thêm Seed URL
frontier.add_url(SEED_URL, depth=0)

print("=== BẮT ĐẦU CRAWL ARXIV ===")
while not frontier.is_empty() and crawled_count < MAX_PAGES:
    current_url, current_depth = frontier.get_next()
    start_time = time.time()

    try:
        response = session.get(current_url, timeout=REQUEST_TIMEOUT)
        elapsed = time.time() - start_time
        status_code = response.status_code
    except requests.RequestException as e:
        failed_requests += 1
        print(f"[Error] {current_url}: {e}")
        continue

    status_code_stats[status_code] = status_code_stats.get(status_code, 0) + 1

    if "text/html" not in response.headers.get("Content-Type", ""):
        continue

    if status_code == 200:
        response.encoding = "utf-8"
        title, authors, raw_abstract, clean_abstract, soup = extract_page_data(response.text, current_url)

        # Tránh trùng lặp nội dung
        content_to_check = clean_abstract if clean_abstract else title
        if detector.is_duplicate(content_to_check):
            print(f"[Duplicate Content Skip] {current_url}")
            continue

        crawled_count += 1
        depth_stats[current_depth] = depth_stats.get(current_depth, 0) + 1
        domain = urlparse(current_url).netloc
        crawled_at = datetime.now().isoformat()

        # Xác định đây là trang bài báo hay trang danh sách
        is_article = 1 if "/abs/" in current_url else 0

        # 1. Lưu vào SQLite theo tên cột mới
        db.insert_page( 
            url=current_url, 
            domain=domain, 
            title=title, 
            abstract=raw_abstract,   # Gắn raw_abstract vào cột abstract
            authors=authors, 
            content=clean_abstract,  # Gắn text xử lý sạch vào cột content
            is_article=is_article,   # Đánh dấu bài báo
            depth=current_depth, 
            status_code=status_code, 
            crawled_at=crawled_at
        )

        # 2. Lấy link và đẩy vào Frontier
        links = extract_and_filter_links(soup, current_url, ALLOWED_DOMAINS)
        for link in links:
            db.insert_link(current_url, link)
            frontier.add_url(link, depth=current_depth + 1)

        print(f"[{crawled_count:03d}] Depth: {current_depth} | Status: {status_code} | Links: {len(links)} | Time: {elapsed:.2f}s | {current_url}")
    else:
        failed_requests += 1

    time.sleep(CRAWL_DELAY)

# In tổng kết
print("\n" + "=" * 25 + " CRAWLING SUMMARY " + "=" * 25)
print(f"Website                : arXiv (Computer Science)")
print(f"Pages Crawled          : {crawled_count}")
print(f"Unique URLs Discovered : {frontier.discovered_count}")
print(f"Skipped URLs           : {frontier.skipped_count}")
print(f"Failed Requests        : {failed_requests}")
for d in range(MAX_DEPTH + 1):
    print(f"  - Depth {d}: {depth_stats.get(d, 0)} pages")
for code, cnt in sorted(status_code_stats.items()):
    print(f"  - HTTP {code}: {cnt}")
print("=" * 68)

=== BẮT ĐẦU CRAWL ARXIV ===
[001] Depth: 0 | Status: 200 | Links: 70 | Time: 0.41s | https://arxiv.org/list/cs/recent
[Duplicate Content Skip] https://arxiv.org/list/cs/new
[002] Depth: 1 | Status: 200 | Links: 8 | Time: 0.06s | https://arxiv.org/abs/2610.03717
[003] Depth: 1 | Status: 200 | Links: 6 | Time: 0.05s | https://arxiv.org/abs/2610.03716
[004] Depth: 1 | Status: 200 | Links: 8 | Time: 0.05s | https://arxiv.org/abs/2610.03715
[005] Depth: 1 | Status: 200 | Links: 11 | Time: 0.04s | https://arxiv.org/abs/2610.03713
[006] Depth: 1 | Status: 200 | Links: 10 | Time: 0.05s | https://arxiv.org/abs/2610.03712
[007] Depth: 1 | Status: 200 | Links: 7 | Time: 0.04s | https://arxiv.org/abs/2610.03710
[008] Depth: 1 | Status: 200 | Links: 6 | Time: 0.05s | https://arxiv.org/abs/2610.03702
[009] Depth: 1 | Status: 200 | Links: 6 | Time: 0.05s | https://arxiv.org/abs/2610.03698
[010] Depth: 1 | Status: 200 | Links: 6 | Time: 0.06s | https://arxiv.org/abs/2610.03695
[011] Depth: 1 | Status:

### 7 Huấn luyện Model & Lưu trực tiếp vào SQLite

In [8]:
# Lấy các cột content (văn bản đã xử lý) của những trang là BÀI BÁO THẬT
cursor = db.conn.cursor()
cursor.execute("SELECT content FROM pages WHERE content IS NOT NULL AND content != '' AND is_article = 1")
corpus = [r[0] for r in cursor.fetchall()]

if len(corpus) >= 2:
    print(f"Đang huấn luyện mô hình TF-IDF trên {len(corpus)} văn bản bài báo...")
    vectorizer = TfidfVectorizer(max_features=3000, stop_words="english")
    vectorizer.fit(corpus)
    vocab_size = len(vectorizer.vocabulary_)
    
    # Đóng gói model nhị phân
    model_bytes = pickle.dumps(vectorizer)
    
    # Lưu vào SQLite
    db.insert_model("arxiv_tfidf_model", "TfidfVectorizer", vocab_size, model_bytes)
    print(f"-> Đã lưu Model thành công vào SQLite (Từ vựng: {vocab_size} từ)!")
else:
    print("Chưa đủ dữ liệu bài báo để huấn luyện model.")

Đang huấn luyện mô hình TF-IDF trên 1530 văn bản bài báo...
-> Đã lưu Model thành công vào SQLite (Từ vựng: 3000 từ)!


### 8 Kiểm tra dữ liệu & Model trong SQLite

In [9]:
# 1. Kiểm tra số dòng
cursor = db.conn.cursor()

cursor.execute("SELECT COUNT(*) FROM pages")
total_pages = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM pages WHERE is_article = 1")
total_articles = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM links")
total_links = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(*) FROM trained_models")
total_models = cursor.fetchone()[0]

print(f"-> Tổng số trang trong DB (tính cả nav): {total_pages}")
print(f"-> Tổng số BÀI BÁO KHOA HỌC chuẩn (is_article = 1): {total_articles}")
print(f"-> Tổng số liên kết trong bảng 'links': {total_links}")
print(f"-> Tổng số models trong bảng 'trained_models': {total_models}\n")

# 2. Hiển thị 5 bài đầu tiên
if total_articles > 0:
    df_pages = pd.read_sql_query(
        "SELECT id, title, authors, abstract, is_article, depth, status_code FROM pages WHERE is_article = 1 LIMIT 5", 
        db.conn
    )
    # Rút gọn abstract cho dễ nhìn giống file mẫu
    df_pages['abstract'] = df_pages['abstract'].apply(lambda x: x[:100] + '...' if x and len(x) > 100 else x)
    display(df_pages)

# 3. Đóng database an toàn
db.close()
print("Đã đóng kết nối cơ sở dữ liệu an toàn.")

-> Tổng số trang trong DB (tính cả nav): 1555
-> Tổng số BÀI BÁO KHOA HỌC chuẩn (is_article = 1): 1530
-> Tổng số liên kết trong bảng 'links': 14141
-> Tổng số models trong bảng 'trained_models': 1



,id,title,authors,abstract,is_article,depth,status_code
0,2,Less Decoder is More Encoder: Geometric Repres...,"Keerthi Kaashyap, Dennis Anthony, Akshay Krish...",This paper examines the role of Novel View Syn...,1,1,200
1,3,MoSE3: Learning World-Space SE(3) at Every Pixel,"Jiahuan Cheng, Zhiyi Li, Tian Xia, Ruojin Cai,...",Dense 3D point tracking has been a prominent p...,1,1,200
2,4,4DCodeBench: Benchmarking Agents on Inverse Gr...,"Ruihong Shen, Žiga Kovačič, Peter Kulits, Xing...","We introduce 4DCodeBench, a benchmark for 4D i...",1,1,200
3,5,What Should World Models Forget? Stratified Re...,"Nishit Anand, Ramani Duraiswami, Dinesh Manocha",Continual learning treats degradation on previ...,1,1,200
4,6,RNADyn: A Benchmark for Generating and Underst...,"Yiming Huang, Lennart Bastian, Hanqun Cao, Lui...",Ribonucleic acid (RNA) functions through confo...,1,1,200


Đã đóng kết nối cơ sở dữ liệu an toàn.
